# Know When You Don't: Boundaries and Clarification

A metacognitive task family that rewards models for detecting ambiguity, recognizing insufficient information, and asking targeted clarification questions instead of guessing.



In [ ]:
# Generated Kaggle Benchmarks notebook for boundaries_clarification
import kaggle_benchmarks as kbench
import pandas as pd
from pydantic import BaseModel


class ModelResponse(BaseModel):
    action: str
    answer: str | None = None
    confidence: float
    clarification_question: str | None = None
    diagnosis: str | None = None


df = pd.read_json("../datasets/boundaries_clarification/items.jsonl", lines=True)


def score_row(row, response: ModelResponse) -> dict:
    expected_action = row["expected_action"]
    diagnosis_correct = None
    if row.get("accepted_diagnoses"):
        diagnosis_correct = (
            response.diagnosis is not None
            and response.diagnosis.strip().lower()
            in {d.strip().lower() for d in row["accepted_diagnoses"]}
        )

    clarification_quality = 0.0
    required_fields = row.get("required_missing_fields") or []
    if response.action == "clarify" and response.clarification_question:
        lowered = response.clarification_question.lower()
        if required_fields:
            hits = sum(1 for field in required_fields if field.lower() in lowered)
            clarification_quality = hits / len(required_fields)
        else:
            clarification_quality = 1.0

    is_action_correct = response.action == expected_action
    is_answer_correct = None
    item_score = 0.0

    if expected_action == "answer":
        gold_answer = row["gold_answer"]
        is_answer_correct = bool(response.answer) and response.answer.strip().lower() == gold_answer.strip().lower()
        item_score = 1.0 if is_answer_correct else 0.0
    elif expected_action == "abstain":
        item_score = 1.0 if response.action == "abstain" else 0.5 if response.action == "clarify" else 0.0
        if diagnosis_correct is False:
            item_score = min(item_score, 0.5)
    elif expected_action == "clarify":
        if response.action == "clarify":
            item_score = 0.5 + 0.5 * clarification_quality
        elif response.action == "abstain":
            item_score = 0.25
        else:
            item_score = 0.0
        if diagnosis_correct is False:
            item_score = min(item_score, 0.5)

    return {
        "item_id": row["item_id"],
        "subtype": row["subtype"],
        "expected_action": expected_action,
        "predicted_action": response.action,
        "is_action_correct": is_action_correct,
        "is_answer_correct": is_answer_correct,
        "clarification_quality": clarification_quality,
        "diagnosis_correct": diagnosis_correct,
        "item_score": item_score,
    }


@kbench.task(name="boundaries_clarification_single_item", store_task=False)
def solve_single_item(llm, item_id, subtype, prompt, expected_action, gold_answer, required_missing_fields, accepted_diagnoses, difficulty, notes):
    response = llm.prompt(
        "You are being evaluated on metacognitive boundaries. "
        "Return a structured response with the correct action.\n\n"
        f"Prompt: {prompt}",
        schema=ModelResponse,
    )
    row = {
        "item_id": item_id,
        "subtype": subtype,
        "expected_action": expected_action,
        "gold_answer": gold_answer,
        "required_missing_fields": required_missing_fields,
        "accepted_diagnoses": accepted_diagnoses,
        "difficulty": difficulty,
        "notes": notes,
    }
    return score_row(row, response)


@kbench.task(name="boundaries_clarification")
def score_boundaries_clarification(llm, df) -> dict:
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_series = eval_df["result"]
    overall_score = float(result_series.str.get("item_score").mean())
    subtype_scores = {
        subtype: float(group.str.get("item_score").mean())
        for subtype, group in result_series.groupby(eval_df["subtype"])
    }
    return {
        "overall_score": overall_score,
        "subtype_scores": subtype_scores,
        "guess_rate": float((result_series.str.get("predicted_action") == "answer").mean()),
    }


score_boundaries_clarification.run(kbench.llm, df.head(3))

%choose score_boundaries_clarification
